# Forecast N-1

Último ciclo de forecast enviado, agrupado por **Segmento**, **Canal** e **Item Principal da Cadeia**, com quantidades previstas totalizadas por soma para **35 meses** de horizonte (`n+1`..`n+35`). O mês do próprio ciclo (`n+0`) é descartado: a primeira coluna de mês é a do mês seguinte ao ciclo.

**Parâmetro:** `forecast_cycle` — data do ciclo (`YYYY-MM-DD`). Deixe **vazio** para usar sempre o ciclo máximo disponível em `raw_forecast_enriched`.

**Fonte:** `parts_hdbk_sandbox.pr_forecast.raw_forecast_enriched` + lookup `pr_cadastrao.material_cadeia`

**Exportações XLSX** (6 arquivos por execução):

| Arquivo | Filtro |
| --- | --- |
| Forecast N-1 2W Doméstico | 2W, canal Doméstico |
| Forecast N-1 2W Exportação | 2W, canal Exportação |
| Forecast N-1 2W | 2W, todos os canais |
| Forecast N-1 4W Doméstico | 4W, canal Doméstico |
| Forecast N-1 4W Exportação | 4W, canal Exportação |
| Forecast N-1 4W | 4W, todos os canais |

**ZIPs:** `Forecast_Previous_Month_HDA_<ciclo>.zip` (2W) · `Forecast_Previous_Month_HAB_<ciclo>.zip` (4W) — ex.: `Forecast_Previous_Month_HDA_2026-09.zip`

**Destino:** `/Volumes/parts_hdbk_sandbox/_file_orchestrator/demand/01_apuracao_demanda/out/`

In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# Configuração
# ---------------------------------------------------------------------------
# Mesma tabela usada na exportação (raw), para o ciclo máximo sempre ter dados
TABLE = "parts_hdbk_sandbox.pr_forecast.raw_forecast_enriched"

# Data máxima do ciclo de forecast disponível na tabela
max_cycle = spark.table(TABLE).agg(F.max("forecast_cycle")).first()[0]
print(f"Data máxima do ciclo de forecast: {max_cycle}")

# Parâmetro editável: ciclo do forecast (YYYY-MM-DD).
# Deixe VAZIO para usar sempre o ciclo máximo disponível na tabela.
# (O default do widget só vale na criação; por isso o fallback é feito aqui.)
dbutils.widgets.text("forecast_cycle", "", "Ciclo do Forecast (vazio = máximo)")
param_cycle = dbutils.widgets.get("forecast_cycle").strip() or str(max_cycle)
print(f"Ciclo selecionado: {param_cycle}")

In [0]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

# ---------------------------------------------------------------------------
# Forecast N-1: Ciclo × Segmento × Canal × Item Principal da Cadeia
# Fonte: raw_forecast_enriched (35 meses de horizonte: n+1..n+35)
# O mês do próprio ciclo (n+0) é descartado: a 1ª coluna é o mês seguinte
# Lookup: material_cadeia para item_principal_cadeia
# ---------------------------------------------------------------------------
RAW_TABLE    = "parts_hdbk_sandbox.pr_forecast.raw_forecast_enriched"
LOOKUP_TABLE = "parts_hdbk_sandbox.pr_cadastrao.material_cadeia"

cycle_date = datetime.strptime(param_cycle, "%Y-%m-%d")
month_labels = [
    (cycle_date + relativedelta(months=i)).strftime("%Y/%m") for i in range(36)
]

# Leitura raw + lookup de cadeia
df_raw = (
    spark.table(RAW_TABLE)
    .filter(F.col("forecast_cycle") == F.lit(param_cycle).cast("date"))
)

df_cadeia = (
    spark.table(LOOKUP_TABLE)
    .select(
        F.col("empresa"),
        F.col("material").alias("_lk_mat"),
        F.col("item_principal_cadeia"),
    )
)

df_joined = (
    df_raw
    .join(
        df_cadeia,
        (F.col("segment") == F.col("empresa"))
        & (F.col("main_material") == F.col("_lk_mat")),
        "left",
    )
    .withColumn(
        "item_principal_cadeia",
        F.coalesce(F.col("item_principal_cadeia"), F.col("main_material")),
    )
    .drop("empresa", "_lk_mat")
)

# Agregação: soma de cada horizonte n+1..n+35 renomeado para mês efetivo
agg_exprs = [
    F.sum(F.col(f"`n+{i}`")).alias(month_labels[i])
    for i in range(1, 36)
]

df_n1 = (
    df_joined
    .withColumn("ciclo", F.date_format("forecast_cycle", "yyyy/MM"))
    .withColumn(
        "segmento",
        F.when(F.col("segment") == "0200", "2W - Motos")
         .when(F.col("segment") == "0500", "4W - Automóveis")
         .otherwise(F.col("segment"))
    )
    .withColumn(
        "canal",
        F.when(F.col("market") == "Domestic", "Doméstico")
         .when(F.col("market") == "Export", "Exportação")
         .otherwise(F.col("market"))
    )
    .groupBy("ciclo", "segmento", "canal", "item_principal_cadeia", "forecast_classification")
    .agg(*agg_exprs)
    .orderBy("segmento", "canal", "item_principal_cadeia")
)

print(f"Ciclo: {param_cycle} | Linhas: {df_n1.count():,}")
display(df_n1)

In [0]:
%pip install openpyxl -q

In [0]:
# ==============================================================================
# EXPORTAÇÃO FORECAST N-1 PARA XLSX
# ==============================================================================
# Célula autocontida: após %pip install o Python reinicia e todas as variáveis
# são perdidas. Por isso reimportamos tudo e reconstruímos o DataFrame aqui.
#
# Saídas (6 arquivos):
#   - Forecast N-1 2W Doméstico <ciclo>.xlsx
#   - Forecast N-1 2W Exportação <ciclo>.xlsx
#   - Forecast N-1 2W <ciclo>.xlsx            (sem distinção de canal)
#   - Forecast N-1 4W Doméstico <ciclo>.xlsx
#   - Forecast N-1 4W Exportação <ciclo>.xlsx
#   - Forecast N-1 4W <ciclo>.xlsx            (sem distinção de canal)
# ==============================================================================
import os
import re
import tempfile
from datetime import datetime
from dateutil.relativedelta import relativedelta

import openpyxl
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# Configuração
# ---------------------------------------------------------------------------
RAW_TABLE    = "parts_hdbk_sandbox.pr_forecast.raw_forecast_enriched"
LOOKUP_TABLE = "parts_hdbk_sandbox.pr_cadastrao.material_cadeia"
VOLUME_PATH  = "/Volumes/parts_hdbk_sandbox/_file_orchestrator/demand/01_apuracao_demanda/out"

# Widget vazio = ciclo máximo disponível na tabela raw
param_cycle = dbutils.widgets.get("forecast_cycle").strip() or str(
    spark.table(RAW_TABLE).agg(F.max("forecast_cycle")).first()[0]
)
cycle_date  = datetime.strptime(param_cycle, "%Y-%m-%d")
cycle_str   = cycle_date.strftime("%Y-%m")

month_labels = [
    (cycle_date + relativedelta(months=i)).strftime("%Y/%m") for i in range(36)
]

# ---------------------------------------------------------------------------
# Reconstrução do DataFrame base (variáveis perdidas após %pip)
# ---------------------------------------------------------------------------
df_raw = (
    spark.table(RAW_TABLE)
    .filter(F.col("forecast_cycle") == F.lit(param_cycle).cast("date"))
)

df_cadeia = (
    spark.table(LOOKUP_TABLE)
    .select(
        F.col("empresa"),
        F.col("material").alias("_lk_mat"),
        F.col("item_principal_cadeia"),
    )
)

df_base = (
    df_raw
    .join(df_cadeia,
          (F.col("segment") == F.col("empresa"))
          & (F.col("main_material") == F.col("_lk_mat")),
          "left")
    .withColumn("item_principal_cadeia",
                F.coalesce(F.col("item_principal_cadeia"), F.col("main_material")))
    .drop("empresa", "_lk_mat")
    .withColumn("ciclo", F.date_format("forecast_cycle", "yyyy/MM"))
    .withColumn("segmento",
        F.when(F.col("segment") == "0200", "2W - Motos")
         .when(F.col("segment") == "0500", "4W - Automóveis")
         .otherwise(F.col("segment")))
    .withColumn("canal",
        F.when(F.col("market") == "Domestic", "Doméstico")
         .when(F.col("market") == "Export", "Exportação")
         .otherwise(F.col("market")))
)

# Horizonte n+1..n+35: o mês do próprio ciclo (n+0) é descartado
agg_exprs = [
    F.sum(F.col(f"`n+{i}`")).alias(month_labels[i])
    for i in range(1, 36)
]

# ---------------------------------------------------------------------------
# Funções auxiliares de exportação (de 6.2_export_demand_to_xlsx_v2)
# ---------------------------------------------------------------------------
_GATILHOS_FORMULA = "=+-@\t\r"
try:
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _CARACTERES_ILEGAIS
except ImportError:
    _CARACTERES_ILEGAIS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


def _neutralizar_formula(valor):
    if isinstance(valor, str) and valor and valor[0] in _GATILHOS_FORMULA:
        return "'" + valor
    return valor


def _valor_seguro(valor):
    if isinstance(valor, str):
        return _CARACTERES_ILEGAIS.sub("", valor)
    if isinstance(valor, (int, float, bool, type(None))):
        return valor
    return _CARACTERES_ILEGAIS.sub("", str(valor))


def _subir_para_volume(caminho_destino, escrever_conteudo):
    descritor, temporario = tempfile.mkstemp(suffix=".xlsx")
    os.close(descritor)
    try:
        escrever_conteudo(temporario)
        with open(temporario, "rb") as arquivo:
            WorkspaceClient().files.upload(caminho_destino, arquivo, overwrite=True)
    finally:
        if os.path.isfile(temporario):
            os.remove(temporario)


def exportar_xlsx(pdf, caminho_destino, nome_aba="Dados"):
    """Exporta um pandas DataFrame para .xlsx no volume."""
    workbook = openpyxl.Workbook()
    planilha = workbook.active
    planilha.title = nome_aba
    planilha.append([_valor_seguro(c) for c in pdf.columns])
    for registro in pdf.itertuples(index=False, name=None):
        planilha.append([_neutralizar_formula(_valor_seguro(v)) for v in registro])
    _subir_para_volume(caminho_destino, workbook.save)


# ---------------------------------------------------------------------------
# Exportação: 6 arquivos
# ---------------------------------------------------------------------------
exports = [
    ("2W - Motos",      "Doméstico",  f"Forecast N-1 2W Doméstico {cycle_str}"),
    ("2W - Motos",      "Exportação", f"Forecast N-1 2W Exportação {cycle_str}"),
    ("2W - Motos",      None,         f"Forecast N-1 2W {cycle_str}"),
    ("4W - Automóveis", "Doméstico",  f"Forecast N-1 4W Doméstico {cycle_str}"),
    ("4W - Automóveis", "Exportação", f"Forecast N-1 4W Exportação {cycle_str}"),
    ("4W - Automóveis", None,         f"Forecast N-1 4W {cycle_str}"),
]

print(f"Exportando forecast ciclo {param_cycle} para:")
print(f"{VOLUME_PATH}\n")

for segmento, canal, nome_arquivo in exports:
    df_filtro = df_base.filter(F.col("segmento") == segmento)

    if canal is not None:
        df_filtro = df_filtro.filter(F.col("canal") == canal)
        group_cols = ["ciclo", "segmento", "canal", "item_principal_cadeia", "forecast_classification"]
    else:
        # Sem distinção de canal: agrega Doméstico + Exportação
        group_cols = ["ciclo", "segmento", "item_principal_cadeia", "forecast_classification"]

    df_agg = (
        df_filtro
        .groupBy(*group_cols)
        .agg(*agg_exprs)
        .orderBy("item_principal_cadeia")
    )

    pdf = df_agg.toPandas()
    caminho = f"{VOLUME_PATH}/{nome_arquivo}.xlsx"
    exportar_xlsx(pdf, caminho)
    print(f"[OK] {nome_arquivo}.xlsx — {len(pdf):,} linhas")

print("\nExportação concluída!")

In [0]:
# ==============================================================================
# CRIAÇÃO DE ARQUIVOS ZIP POR SEGMENTO
# ==============================================================================
# Agrupa os .xlsx exportados pelo prefixo do segmento e gera um ZIP para cada:
#   - Forecast_Previous_Month_HDA_<ciclo>.zip  →  arquivos 2W (Motos)
#   - Forecast_Previous_Month_HAB_<ciclo>.zip  →  arquivos 4W (Automóveis)
# Apenas os .xlsx do ciclo exportado (cycle_str) são incluídos.
# ==============================================================================
import zipfile

# Mapeamento segmento → prefixo do ZIP
SEGMENTO_ZIP = {
    "2W": "Forecast_Previous_Month_HDA",
    "4W": "Forecast_Previous_Month_HAB",
}

print("=" * 70)
print("CRIANDO ARQUIVOS ZIP POR SEGMENTO")
print("=" * 70)

# Agrupar os .xlsx do Forecast N-1 do ciclo exportado por segmento
arquivos_xlsx = [
    item for item in dbutils.fs.ls(VOLUME_PATH)
    if item.name.startswith("Forecast N-1") and item.name.endswith(f" {cycle_str}.xlsx")
]

grupos = {"2W": [], "4W": []}
for item in arquivos_xlsx:
    if " 2W " in item.name:
        grupos["2W"].append(item.name)
    elif " 4W " in item.name:
        grupos["4W"].append(item.name)

for segmento, arquivos in grupos.items():
    if not arquivos:
        print(f"\n[AVISO] Nenhum .xlsx encontrado para {segmento} — ZIP não criado.")
        continue

    nome_zip = f"{SEGMENTO_ZIP[segmento]}_{cycle_str}.zip"
    caminho_zip_local = os.path.join(tempfile.gettempdir(), nome_zip)
    caminho_zip_volume = f"{VOLUME_PATH}/{nome_zip}"

    with zipfile.ZipFile(caminho_zip_local, "w", zipfile.ZIP_DEFLATED) as zf:
        for nome_xlsx in sorted(arquivos):
            caminho_xlsx = f"{VOLUME_PATH}/{nome_xlsx}"
            zf.write(caminho_xlsx, arcname=nome_xlsx)

    with open(caminho_zip_local, "rb") as f:
        WorkspaceClient().files.upload(caminho_zip_volume, f, overwrite=True)

    os.remove(caminho_zip_local)

    print(f"\n[OK] {nome_zip}: {len(arquivos)} arquivos empacotados")
    for a in sorted(arquivos):
        print(f"     • {a}")

print("\n" + "=" * 70)
print("RESUMO")
print("=" * 70)
for segmento, prefixo in SEGMENTO_ZIP.items():
    n = len(grupos[segmento])
    status = f"{n} arquivos" if n else "não criado"
    print(f"  {prefixo}_{cycle_str}.zip → {status}")
print(f"\nLocal: {VOLUME_PATH}")